In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_json('./analysis/data/pilot-results.json')
index = df.index

df = pd.json_normalize(df['readabilityVisualComfort'], meta=df.index.name)
df.index = index

df.head()

In [ ]:
df.info()

In [ ]:
# throw out last row, full empty
df = df.iloc[:-1]

In [ ]:
# get rid of lists in binary columns
binary_columns = ['academicReading', 'graphicDesign']
for col in binary_columns:
    df[col] = df[col].apply(lambda x: 'yes' if isinstance(x, list) and 'yes' in x else 'no')




In [ ]:
sns.countplot(x='nativeLanguage', data=df)

In [ ]:
# from otherLanguages make a new column otherLanguageCount with the number of other languages
df['otherLanguageCount'] = df['otherLanguages'].apply(lambda x: len(x) if isinstance(x, list) else 0)

In [ ]:
sns.countplot(x='otherLanguageCount', data=df)

In [ ]:
df['otherLanguages'].value_counts()

In [ ]:
# Ensure 'booksRead' is treated as a numeric column
df['booksRead'] = pd.to_numeric(df['booksRead'], errors='coerce', downcast='integer')

# Sort the x-axis by the number of books read
sorted_order = df['booksRead'].sort_values().unique()

sns.countplot(x='booksRead', data=df, order=sorted_order)

In [ ]:
sns.countplot(x='booksTimeframe', data=df)

In [ ]:
df['everydayReading'].value_counts()

In [ ]:
df['everydayReadingOther'].value_counts()

In [ ]:
df['graphicDesign'].value_counts()

In [ ]:
sns.countplot(x='graphicDesign', data=df)

In [ ]:
df.columns

In [ ]:
df_slider = df.copy(deep=True)
# drop all rows where sliderSettings is NaN
df_slider = df_slider[pd.notna(df_slider['sliderSettings'])]

df_select = df.copy(deep=True)
# drop all rows where selectedLanguages is NaN
df_select = df_select[pd.notna(df_select['selectedLanguages'])]

print(f'Total rows: {df.shape[0]}')
print(f'Total rows (slider): {df_slider.shape[0]}')
print(f'Total rows (select): {df_select.shape[0]}')


## A look at actual test results

##### Load language data

In [ ]:
import json
language_map = json.loads(open('./analysis/pilot-languages.json').read())
language_map = {lang: group for group, group_languages in language_map.items() for lang in group_languages}

# look up a group with a language, for example: language_map['English']

### Slider test

In [ ]:
default_slider = {
    'lineHeight': 1.2,
    'letterSpacing': 0.0,
    'wordSpacing': 0.0,
}

In [ ]:
df_slider_results = pd.DataFrame(columns=["responseId", "nativeLanguage", "otherLanguages", "shownLanguage", "round", "sameGroup", "sameLanguage","lineHeight_difference", "letterSpacing_difference", "wordSpacing_difference"])

In [ ]:
for i, row in df_slider.iterrows():
    results_row = pd.DataFrame(columns=["responseId", "nativeLanguage", "otherLanguages", "shownLanguage", "round", "sameGroup", "sameLanguage", "lineHeight_difference", "letterSpacing_difference", "wordSpacing_difference"], data=[[row.name, row['nativeLanguage'], row['otherLanguages'], "", np.nan, False, False, np.nan, np.nan, np.nan]])
    native_lang = row['nativeLanguage']
    questions = row['sliderSettings']
    for question in questions:
        difference_map = {
            'lineHeight': 0,
            'letterSpacing': 0,
            'wordSpacing': 0,
        }
        for val in default_slider.keys():
            value = float(question[val])
            if default_slider[val] != value:
                difference_map[val] = value - default_slider[val]
        for k, v in difference_map.items():
            results_row[f'{k}_difference'] = v

        results_row['shownLanguage'] = question['language']
        results_row['round'] = question['round']
        results_row['sameGroup'] = (language_map.get(question['language']) == language_map.get(native_lang))
        results_row['sameLanguage'] = (question['language'] == native_lang)
        df_slider_results = pd.concat([df_slider_results, results_row], ignore_index=True)


df_slider_results.head()
        


### Word Spacing
##### Native language vs not native language

In [ ]:
sns.boxplot(x=df_slider_results['sameLanguage'], y=df_slider_results['wordSpacing_difference'], hue=df_slider_results['sameLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.stripplot(x=df_slider_results['sameLanguage'], y=df_slider_results['wordSpacing_difference'], hue=df_slider_results['sameLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
df_slider_results[df_slider_results['sameLanguage']==False]['wordSpacing_difference'].describe()

In [ ]:
df_slider_results[df_slider_results['sameLanguage']==True]['wordSpacing_difference'].describe()

##### Same group vs not same group

In [ ]:
sns.boxplot(x=df_slider_results['sameGroup'], y=df_slider_results['wordSpacing_difference'], hue=df_slider_results['sameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.stripplot(x=df_slider_results['sameGroup'], y=df_slider_results['wordSpacing_difference'], hue=df_slider_results['sameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### Line height
Default = 1.2

In [ ]:
sns.boxplot(x=df_slider_results['sameLanguage'], y=df_slider_results['lineHeight_difference'], hue=df_slider_results['sameLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.stripplot(x=df_slider_results['sameLanguage'], y=df_slider_results['lineHeight_difference'], hue=df_slider_results['sameLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.boxplot(x=df_slider_results['sameGroup'], y=df_slider_results['lineHeight_difference'], hue=df_slider_results['sameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.stripplot(x=df_slider_results['sameGroup'], y=df_slider_results['lineHeight_difference'], hue=df_slider_results['sameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

##### Letter spacing

In [ ]:
sns.boxplot(x=df_slider_results['sameLanguage'], y=df_slider_results['letterSpacing_difference'], hue=df_slider_results['sameLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.stripplot(x=df_slider_results['sameLanguage'], y=df_slider_results['letterSpacing_difference'], hue=df_slider_results['sameLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.boxplot(x=df_slider_results['sameGroup'], y=df_slider_results['letterSpacing_difference'], hue=df_slider_results['sameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.stripplot(x=df_slider_results['sameGroup'], y=df_slider_results['letterSpacing_difference'], hue=df_slider_results['sameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
df_slider_results.to_csv('./analysis/data/pilot-slider-results-processed.csv', index=False)

### Select test

In [ ]:
df_select_results = pd.DataFrame(columns=["responseId", "nativeLanguage", "otherLanguages", "round", "firstSelectedLanguage", "secondSelectedLanguage", "selectedMatch", "ifMatchSameGroupAsNative", "nativeInSelection"])

In [ ]:
# Add Q'eqchi' to group3: never seen before, related in some ways to spanish speaking?
language_map["Q'eqchi'"] = "Group3"

In [ ]:
for i, row in df_select.iterrows():
    results_row = pd.DataFrame(columns=["responseId", "nativeLanguage", "otherLanguages", "round", "firstSelectedLanguage", "secondSelectedLanguage", "selectedMatch", "ifMatchSameGroupAsNative", "nativeInSelection"], data=[[row.name, row['nativeLanguage'], row['otherLanguages'], np.nan, "", "", False, False, False]])
    native_lang = row['nativeLanguage']
    questions = row['selectedLanguages']
    for question in questions:
        # Somehow there are questions with less than 2 selections or even ZERO selections??
        if 'selections' not in question or len(question['selections']) < 2:
            continue
        try:
            first_lang = question['selections'][0]['language']
            second_lang = question['selections'][1]['language']
            results_row['firstSelectedLanguage'] = first_lang
            results_row['secondSelectedLanguage'] = second_lang
        except:
            print(row.name, question)

        # print(first_lang, second_lang, native_lang)

        
        # Edge case
        if first_lang == 'Haitian Creole':
            first_lang = 'Creole'

        if second_lang == 'Haitian Creole':
            second_lang = 'Creole'

        selected_match = (language_map[first_lang] == language_map[second_lang])
        results_row['selectedMatch'] = selected_match


        if selected_match:
            results_row['ifMatchSameGroupAsNative'] = (language_map.get(first_lang) == language_map.get(native_lang))

        if native_lang == first_lang or native_lang == second_lang:
            results_row['nativeInSelection'] = True

        df_select_results = pd.concat([df_select_results, results_row], ignore_index=True)


df_select_results.head()


In [ ]:
sns.countplot(x=df_select_results['selectedMatch'])

In [ ]:
sns.countplot(x=df_select_results['selectedMatch'], hue=df_select_results['nativeInSelection'])

In [ ]:
sns.countplot(x=df_select_results['selectedMatch'], hue=df_select_results['ifMatchSameGjuproupAsNative'])

In [ ]:
# save to csv
df_select_results.to_csv('./analysis/data/pilot-select-results-processed.csv', index=False)